# TF-IDF + NMF Topic Modeling

Runs on the segmented postings dataset, split into AI-mention / non-AI-mention
groups. Two independent text-analysis views: NMF topic modeling, and raw
TF-IDF term-score comparison between groups.

**Input:** `data/mia_postings_final2_fixed2.csv`
**Outputs:** `data/mia_tfidf_nmf_topic_terms.csv`, `data/mia_tfidf_term_scores.csv`,
`data/mia_tfidf_group_diff.csv`

In [ ]:
CSV_PATH = "data/mia_postings_final2_fixed2.csv"
TEXT_COL = "description"                  # column with the job posting text
AI_FLAG_COL = "mentions_ai"               # boolean/0-1 column: AI-mention vs not
LANG_FILTER = True                        # drop non-English postings before modeling
MIN_VERBATIM_COUNT = 5                    # strip sentences repeated >= this many times (boilerplate)
N_TOPICS = 5                              # topics per group
NGRAM_RANGE = (1, 2)                      # unigram + bigram
MAX_FEATURES = 5000                       # cap vocabulary size
TOP_WORDS_PER_TOPIC = 12
OUTPUT_CSV = "data/mia_tfidf_nmf_topic_terms.csv"

In [ ]:
import re
import pandas as pd
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [ ]:
try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0  # deterministic langdetect
    LANGDETECT_AVAILABLE = True
except ImportError:
    LANGDETECT_AVAILABLE = False
    print("langdetect not installed — run: pip install langdetect --break-system-packages")

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
assert TEXT_COL in df.columns, f"'{TEXT_COL}' not found — check CONFIG.TEXT_COL against: {list(df.columns)}"
assert AI_FLAG_COL in df.columns, f"'{AI_FLAG_COL}' not found — check CONFIG.AI_FLAG_COL against: {list(df.columns)}"

df = df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)
print(f"{len(df):,} rows with non-null {TEXT_COL}")

In [ ]:
if LANG_FILTER and LANGDETECT_AVAILABLE:
    def is_english(text: str) -> bool:
        try:
            return detect(text[:1000]) == "en"  # first 1000 chars is enough, and much faster
        except Exception:
            return False

    before = len(df)
    df["_is_en"] = df[TEXT_COL].apply(is_english)
    df = df[df["_is_en"]].drop(columns="_is_en")
    print(f"English filter: {before:,} -> {len(df):,} rows")
else:
    print("Skipping language filter (disabled or langdetect unavailable)")

In [ ]:
_SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")

In [ ]:
# Sentences repeated verbatim across many postings (EEO statements, benefits
# blurbs, boilerplate "About the company" text) drown out real topic signal.
# Any sentence repeated >= MIN_VERBATIM_COUNT times across the corpus is
# dropped before vectorizing.
def split_sentences(text: str) -> list[str]:
    return [s.strip() for s in _SENTENCE_SPLIT.split(text) if s.strip()]

print("Counting sentence frequency across corpus (this is the slow-ish step)...")
sentence_counts = Counter()
doc_sentences = df[TEXT_COL].apply(split_sentences)
for sentences in doc_sentences:
    sentence_counts.update(set(sentences))  # count each sentence once per posting

In [ ]:
boilerplate = {s for s, c in sentence_counts.items() if c >= MIN_VERBATIM_COUNT}
print(f"Found {len(boilerplate):,} boilerplate sentences (repeated >= {MIN_VERBATIM_COUNT}x)")

In [ ]:
def strip_boilerplate(sentences: list[str]) -> str:
    return " ".join(s for s in sentences if s not in boilerplate)

In [ ]:
df["_clean_text"] = doc_sentences.apply(strip_boilerplate)
df = df[df["_clean_text"].str.len() > 0]
print(f"{len(df):,} rows remain after stripping boilerplate")

In [ ]:
# Added on top of sklearn's built-in English stopword list — EEO/benefits
# language that survives the boilerplate-sentence filter because it appears
# in slightly different sentences each time.
EEO_BENEFITS_STOPWORDS = [
    "equal", "opportunity", "employer", "disability", "veteran", "race", "religion",
    "sex", "gender", "orientation", "identity", "protected", "eoe", "affirmative",
    "action", "diversity", "inclusion", "accommodation", "applicants", "applicant",
    "benefits", "benefit", "insurance", "dental", "vision", "401k", "pto",
    "paid", "time", "off", "salary", "compensation", "range", "bonus", "equity",
    "healthcare", "health", "wellness", "employment", "employees", "employee",
    "ll", "ve", "re", "don", "didn",  # contraction fragments (we'll, we've, etc. losing the apostrophe)
]

In [ ]:
from sklearn.feature_extraction import text as sk_text
custom_stopwords = list(sk_text.ENGLISH_STOP_WORDS.union(EEO_BENEFITS_STOPWORDS))

In [ ]:
ai_mask = df[AI_FLAG_COL].astype(bool)
group_ai = df.loc[ai_mask, "_clean_text"]
group_non_ai = df.loc[~ai_mask, "_clean_text"]
print(f"AI-mention group: {len(group_ai):,} postings")
print(f"Non-AI-mention group: {len(group_non_ai):,} postings")

In [ ]:
def run_tfidf_nmf(texts: pd.Series, label: str, n_topics: int = N_TOPICS):
    vectorizer = TfidfVectorizer(
        max_features=MAX_FEATURES,
        ngram_range=NGRAM_RANGE,
        stop_words=custom_stopwords,
        min_df=5,       # ignore terms in fewer than 5 postings
        max_df=0.6,     # ignore terms in more than 60% of postings (still-too-common boilerplate)
    )
    tfidf_matrix = vectorizer.fit_transform(texts)
    print(f"[{label}] TF-IDF matrix: {tfidf_matrix.shape[0]:,} docs x {tfidf_matrix.shape[1]:,} terms")

    nmf = NMF(n_components=n_topics, random_state=42, init="nndsvda", max_iter=500)
    nmf.fit(tfidf_matrix)

    feature_names = vectorizer.get_feature_names_out()
    rows = []
    for topic_idx, topic in enumerate(nmf.components_):
        top_indices = topic.argsort()[::-1][:TOP_WORDS_PER_TOPIC]
        top_terms = [feature_names[i] for i in top_indices]
        rows.append({
            "group": label,
            "topic": topic_idx + 1,
            "top_terms": ", ".join(top_terms),
        })
        print(f"[{label}] Topic {topic_idx + 1}: {', '.join(top_terms)}")

    topic_df = pd.DataFrame(rows)
    # return the matrix + vectorizer too, so raw TF-IDF scores are available outside this function
    return topic_df, tfidf_matrix, vectorizer

In [ ]:
results_ai, tfidf_matrix_ai, vectorizer_ai = run_tfidf_nmf(group_ai, "AI-mention")

In [ ]:
results_non_ai, tfidf_matrix_non_ai, vectorizer_non_ai = run_tfidf_nmf(group_non_ai, "Non-AI-mention")

In [ ]:
results = pd.concat([results_ai, results_non_ai], ignore_index=True)
results.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(results)} topic rows to {OUTPUT_CSV}")
results

## Raw TF-IDF term scores (independent of NMF topics)

TF-IDF gives a score per (document, term) pair, not one number per word. To
rank terms for a whole group, average each term's score across all postings
in that group — terms consistently important score higher than terms that
spike hard in just one or two postings.

Note: raw within-group ranking mostly fails to discriminate between groups —
18 of the top 25 terms are identical between AI-mention and non-AI-mention
(digital, customer, product, market, reporting...). The group-diff comparison
further down is the more informative view.

In [ ]:
def top_tfidf_terms(tfidf_matrix, vectorizer, label, top_n=25):
    feature_names = vectorizer.get_feature_names_out()
    mean_scores = tfidf_matrix.mean(axis=0).A1  # average TF-IDF per term across all docs in the group
    term_scores = pd.DataFrame({
        "term": feature_names,
        "mean_tfidf": mean_scores,
    }).sort_values("mean_tfidf", ascending=False).head(top_n)
    term_scores.insert(0, "group", label)
    return term_scores.reset_index(drop=True)

In [ ]:
tfidf_scores_ai = top_tfidf_terms(tfidf_matrix_ai, vectorizer_ai, "AI-mention")

In [ ]:
tfidf_scores_non_ai = top_tfidf_terms(tfidf_matrix_non_ai, vectorizer_non_ai, "Non-AI-mention")

In [ ]:
tfidf_scores = pd.concat([tfidf_scores_ai, tfidf_scores_non_ai], ignore_index=True)
tfidf_scores.to_csv("data/mia_tfidf_term_scores.csv", index=False)
print(f"Saved {len(tfidf_scores)} term-score rows to mia_tfidf_term_scores.csv")
tfidf_scores

## Group-diff comparison

Compares mean TF-IDF *differences* between groups rather than within-group
rankings — this is what actually separates AI-mention from non-AI-mention
vocabulary. See Inferences.md, "Text analysis — TF-IDF term scores" for the
full interpretation, including the `tools` = 0.0 filtering artifact noted in
the methodological caveats.

In [ ]:
def full_mean_tfidf(tfidf_matrix, vectorizer):
    feature_names = vectorizer.get_feature_names_out()
    mean_scores = tfidf_matrix.mean(axis=0).A1
    return pd.DataFrame({"term": feature_names, "mean_tfidf": mean_scores})

In [ ]:
ai_scores = full_mean_tfidf(tfidf_matrix_ai, vectorizer_ai)
non_ai_scores = full_mean_tfidf(tfidf_matrix_non_ai, vectorizer_non_ai)

In [ ]:
# The two vectorizers were fit independently per group, so a term missing
# from one group's vocabulary is treated as a score of 0 here. Reasonable
# approximation, not a perfectly controlled comparison — see Inferences.md
# methodological caveats.
comparison = ai_scores.merge(
    non_ai_scores, on="term", how="outer", suffixes=("_ai", "_non_ai")
).fillna(0)

comparison["diff"] = comparison["mean_tfidf_ai"] - comparison["mean_tfidf_non_ai"]

In [ ]:
top_ai_leaning = comparison.sort_values("diff", ascending=False).head(25)

In [ ]:
top_non_ai_leaning = comparison.sort_values("diff", ascending=True).head(25)

In [ ]:
print("Most AI-mention-skewed terms:")
print(top_ai_leaning[["term", "mean_tfidf_ai", "mean_tfidf_non_ai", "diff"]].to_string(index=False))

In [ ]:
print("\nMost non-AI-mention-skewed terms:")
print(top_non_ai_leaning[["term", "mean_tfidf_ai", "mean_tfidf_non_ai", "diff"]].to_string(index=False))

In [ ]:
comparison.sort_values("diff", ascending=False).to_csv("data/mia_tfidf_group_diff.csv", index=False)